# sim_dual_arm — 本机 rclone 同步数据集与模型

本 Notebook 在**本机 Python 内核**运行，核心同步逻辑位于 [`docs/colab/scripts/drive_sync.py`](scripts/drive_sync.py)，直接复用本机已配置好的 `rclone`（无需再次登录 Google Drive）。

## 使用流程

1. 在右上角 **Select Kernel** 选择**本机 Python 3 环境**。
2. 修改下方 **第 1 节** 的 `ROBOT`、`BACKEND` 与 `TASK`（例如 `marvin` / `mujoco` / `handover` 或 `marvin` / `mujoco` / `stack`）并运行。
3. 按需要单独运行后续功能格：
   - **第 2 节**：查看本地与云端数据集、模型及训练日志清单；
   - **第 3 节**：将本地 `output/<robot>/<backend>/<task>/datasets/lerobot` 打包并上传到 Google Drive；
   - **第 4 节**：从 Google Drive 下载数据集回本地；
   - **第 5 节**：从 Google Drive 下载指定或最新训练模型与日志回本地；
   - **第 6 节**：将本地模型与日志上传备份至 Google Drive。


In [ ]:
# 1. 参数配置与环境初始化（修改 ROBOT / BACKEND / TASK 后运行）
import sys
from pathlib import Path

ROBOT = "marvin"
BACKEND = "mujoco"
TASK = "handover"
RCLONE_REMOTE = "gdrive:sim_dual_arm"
DATASET_ARCHIVE = "lerobot.tar"

def _ensure_repo_on_path() -> Path:
    origin = Path.cwd().resolve()
    for candidate in (origin, *origin.parents):
        if (candidate / ".git").exists() and (candidate / "shared" / "scenarios").is_dir():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise FileNotFoundError(f"未从路径向上找到仓库根目录: {origin}")

REPO_ROOT = _ensure_repo_on_path()
from docs.colab.scripts.drive_sync import (
    create_context,
    download_dataset,
    download_model,
    show_status,
    upload_dataset,
    upload_model,
)

sync_ctx = create_context(
    robot=ROBOT,
    backend=BACKEND,
    task=TASK,
    rclone_remote=RCLONE_REMOTE,
    dataset_archive=DATASET_ARCHIVE,
    repo_root=REPO_ROOT,
)

In [ ]:
# 2. 查看本地与云端状态（只读）
show_status(sync_ctx)

In [ ]:
# 3. 上传数据集（本地 output/<robot>/<backend>/<task>/datasets/lerobot -> Google Drive）
upload_dataset(sync_ctx)

In [ ]:
# 4. 下载数据集（Google Drive -> 本地 output/<robot>/<backend>/<task>/datasets/lerobot）
OVERWRITE_LOCAL = False  # 为防止误覆盖本地数据集，需覆盖时改为 True
download_dataset(sync_ctx, overwrite_local=OVERWRITE_LOCAL)

In [ ]:
# 5. 下载模型与训练日志（Google Drive -> 本地 output/<robot>/<backend>/<task>/models/）
MODEL_NAME = "latest"       # "latest" 自动选云端最新 act_*，也可填写具体目录名
SYNC_TRAINING_LOGS = True   # True 时同步拉取对应 logs/training/<model>/
OVERWRITE_LOCAL = False     # 若本地已存在同名模型目录，需覆盖时改为 True

download_model(
    sync_ctx,
    model_name=MODEL_NAME,
    sync_training_logs=SYNC_TRAINING_LOGS,
    overwrite_local=OVERWRITE_LOCAL,
)

In [ ]:
# 6. 上传本地模型与日志备份到 Google Drive
MODEL_NAME = "latest"      # "latest" 自动选本地最新 act_*，也可填写具体目录名
SYNC_TRAINING_LOGS = True  # True 时同步上传对应 logs/training/<model>/

upload_model(
    sync_ctx,
    model_name=MODEL_NAME,
    sync_training_logs=SYNC_TRAINING_LOGS,
)